# 5.1 - Adult Counterfactual Sampling using FACE

This notebook implements the **Feasible and Actionable Counterfactual Explanations (FACE)** method (Poyiadzi et al., 2020) as a baseline comparison for the Certified Atlas on the Adult dataset.

FACE constructs a high-density graph of the data points and finds the shortest path from the factual instance to the target class, ensuring the counterfactual lies on the data manifold.

In [1]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import torch
import pandas as pd
import networkx as nx
from sklearn.neighbors import kneighbors_graph
from sklearn.metrics import pairwise_distances
from torch.utils.data import TensorDataset

from training.lit_classifier import LitClassifier
from training.datamodules.adult import AdultDataModule, _ALL_COLS, INPUT_TYPES, _CATEGORICAL, _NUMERICAL, CARDINALITIES
from models.classifiers import TabularClassifier

DEVICE = 'cpu'
CKPT  = '../checkpoints/adult_classifier/last-v3.ckpt'

In [2]:
# Load the model architecture explicitly
tabular_model = TabularClassifier(
    input_types=INPUT_TYPES,
    cardinalities=CARDINALITIES,
    embedding_dim=1,
    hidden_dims=[64, 32],
    num_classes=2,
)

lit = LitClassifier.load_from_checkpoint(CKPT, model=tabular_model, map_location=DEVICE)
model = lit.model.eval().to(DEVICE)

print(f'embed_dim : {model.embed_dim}')
print(f'net       : {model.net}')

embed_dim : 14
net       : Sequential(
  (0): Linear(in_features=14, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=2, bias=True)
)


In [3]:
# Load the data
dm = AdultDataModule(data_dir='../data/', batch_size=256, num_workers=0)
dm.setup()

X_train, y_train = dm.train_ds.tensors
X_test,  y_test  = dm.test_ds.tensors
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: torch.Size([39074, 14]), Test: torch.Size([4884, 14])


In [4]:
# Calculate Embeddings for the training set (to build the manifold graph)
with torch.no_grad():
    Z_train = model.embed(X_train.to(DEVICE)).cpu().numpy()
    y_train_np = y_train.cpu().numpy()

print(f'Embedded training set: {Z_train.shape}')

Embedded training set: (39074, 14)


## Build FACE k-NN Graph
We build a weighted k-Nearest Neighbors graph using the dense embedding space. The shortest path on this graph ensures the generated counterfactual respects the training data distribution.

In [5]:
def build_face_graph(Z, k=15):
    """Builds the k-NN graph for FACE and returns a NetworkX graph."""
    print(f"Building {k}-NN graph...")
    A = kneighbors_graph(Z, n_neighbors=k, mode='distance', include_self=False)
    G = nx.from_scipy_sparse_array(A)
    return G

G_train = build_face_graph(Z_train, k=15)

Building 15-NN graph...


In [6]:
def face_counterfactual(Z_query, graph, Z_data, y_data, target_class, k=15):
    """
    Finds a tabular counterfactual using the FACE methodology.
    
    1. Identify the nearest node in the data graph to the query.
    2. Find the shortest path from this node to ANY node of the target_class.
    """
    # 1. Find the closest training node to our query
    dists = pairwise_distances(Z_query.reshape(1, -1), Z_data)[0]
    start_node = np.argmin(dists)
    
    # Note: If the start_node is already the target class, we don't need a counterfactual.
    if y_data[start_node] == target_class:
        return Z_data[start_node], [start_node]

    # 2. Extract target nodes (nodes belonging to the target class)
    target_nodes = set(np.where(y_data == target_class)[0])
    
    # 3. Find shortest path
    # We can use NetworkX's multi-source Dijkstra or individually check paths (costly).
    # A more efficient way: compute single-source shortest paths from start_node to all nodes
    lengths, paths = nx.single_source_dijkstra(graph, start_node, weight='weight')
    
    best_target = None
    best_dist = float('inf')
    
    for node, dist in lengths.items():
        if node in target_nodes and dist < best_dist:
            best_dist = dist
            best_target = node
            
    if best_target is None:
        raise ValueError(f"No path found to class {target_class}.")
        
    path_nodes = paths[best_target]
    # The actual physical counterfactual point is the coordinates of the target node.
    Z_cf = Z_data[best_target]
    
    return Z_cf, path_nodes

## Find a FACE Counterfactual

In [7]:
# Let's pick a test sample that was predicted as class 0 (Low Income), 
# and we want to find a counterfactual in class 1 (High Income).

with torch.no_grad():
    logits = model(X_test.to(DEVICE))
    y_pred = logits.argmax(dim=1).cpu()

idx = torch.where(y_pred == 0)[0][0].item()
x_fact = X_test[idx]

with torch.no_grad():
    z_fact = model.embed(x_fact.unsqueeze(0).to(DEVICE)).cpu().numpy()[0]

print(f"Finding FACE counterfactual for sample {idx} (predicts 0, generating for 1)...")
z_cf_face, path_indices = face_counterfactual(
    Z_query=z_fact,
    graph=G_train,
    Z_data=Z_train,
    y_data=y_train_np,
    target_class=1
)

print(f"Original prediction: 0")
with torch.no_grad():
    cf_pred = model.net(torch.tensor(z_cf_face, dtype=torch.float32).unsqueeze(0).to(DEVICE)).argmax().item()
print(f"FACE CF prediction: {cf_pred}")
print(f"Path length: {len(path_indices)} steps through data manifold.")

Finding FACE counterfactual for sample 0 (predicts 0, generating for 1)...
Original prediction: 0
FACE CF prediction: 0
Path length: 4 steps through data manifold.


## Quantitative Evaluation Framework

Now we run the generated FACE counterfactual through our generalized counterfactual metric evaluator. This computes the continuous + categorical specific distances alongside $k$-NN Density scores.

In [9]:
from preimage_sampling.metrics import TabularCounterfactualEvaluator

# _NUMERICAL and _CATEGORICAL from AdultDataModule are sets/lists of string names.
# We must convert them to integer indices representing their column position for numpy arrays to work.
num_col_indices = [i for i, col in enumerate(_ALL_COLS) if col in _NUMERICAL]
cat_col_indices = [i for i, col in enumerate(_ALL_COLS) if col in _CATEGORICAL]

evaluator = TabularCounterfactualEvaluator(
    model=model,
    X_train_tensor=X_train,
    num_cols=num_col_indices,
    cat_cols=cat_col_indices,
    device=DEVICE
)

# Recover the raw factual configuration (x)
x_original = x_fact

# Recover the RAW generated counterfactual (x_cf). 
# For FACE, the counterfactual strictly corresponds to a physical training node.
target_idx = path_indices[-1]
x_cf_face_raw = X_train[target_idx]

# Run standard evaluation protocol for standard tabular dataset properties
metrics = evaluator.evaluate(
    x_orig=x_original, 
    x_cf=x_cf_face_raw, 
    target_class=1
)

print("Evaluation Metrics for FACE:")
for key, value in metrics.items():
    print(f"  {key:25s}: {value:.4f}")

Evaluation Metrics for FACE:
  validity                 : 0.0000
  predicted_class          : 0.0000
  L0_distance              : 5.0000
  L1_distance              : 5.3860
  L2_distance              : 2.4182
  plausibility_knn_5_dist  : 0.2255
  plausibility_lof_score   : -1.0128
